# 10: Sinais de anomalia e os guards do Supervisor

Dois mecanismos em **código**, sem LLM, que tornam o pipeline previsível:

1. **Sinais de anomalia**: regras objetivas calculadas com os números da empresa logo depois do YFinance.
2. **Guards**: verificações no início do `supervisor_node` que decidem o próximo passo antes de chamar o LLM.

Por que em código? Antes, quem decidia se havia anomalia era o LLM, lendo o texto. Às vezes ele não via um
lucro caindo 60%; às vezes pulava a etapa do setor. Uma regra objetiva não esquece, não custa chamada e
pode ser testada.

In [1]:
import sys
import warnings
from pathlib import Path

# Sobe pastas até achar a raiz do projeto (onde está o pyproject.toml), de onde quer que o notebook seja aberto
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(RAIZ))

from dotenv import load_dotenv
_ = load_dotenv(RAIZ / ".env")

# Aviso de depreciação interno do LangGraph, sem relação com o projeto: só polui a saída.
# O filtro vem depois do import do langchain_core, que instala o filtro dele ao ser importado.
from langchain_core._api.deprecation import LangChainPendingDeprecationWarning
warnings.filterwarnings("ignore", category=LangChainPendingDeprecationWarning)
warnings.filterwarnings("ignore", message="Function wrap_gemini is in beta")

## 1. As regras de anomalia

| Sinal | Regra | Por que importa |
|---|---|---|
| `queda_lucro` | lucro do último trimestre mais de 30% abaixo do **mesmo trimestre do ano anterior** | comparar com o ano anterior tira o efeito da sazonalidade (um 1º trimestre fraco todo ano não é anomalia) |
| `prejuizo` | lucro do último trimestre negativo | a empresa perdeu dinheiro no período |
| `piotroski_fraco` | Piotroski de 3 ou menos | fundamentos fracos em vários critérios ao mesmo tempo |

Os limites são constantes no topo de `tools/anomalias.py`. Dado faltando faz a regra ser pulada, e qualquer
erro devolve lista vazia: o detector nunca derruba o pipeline.

In [2]:
from fluxo_financeiro.tools.anomalias import PIOTROSKI_FRACO, QUEDA_LUCRO_LIMITE, detectar_sinais_anomalia

print(f"Limite de queda: {QUEDA_LUCRO_LIMITE:.0%} | Piotroski fraco: <= {PIOTROSKI_FRACO}\n")
sinais = {}
for ticker in ("VALE3", "WEGE3"):
    sinais[ticker] = detectar_sinais_anomalia(ticker)
    print(f"{ticker}: {len(sinais[ticker])} sinal(is)")
    for s in sinais[ticker]:
        print(f"   [{s.tipo}] {s.descricao}")

2026-11-06 21:39:00,312 - fluxo_financeiro - INFO - [anomalias] detectando sinais para VALE3.SA


Limite de queda: 30% | Piotroski fraco: <= 3



2026-11-06 21:39:01,617 - fluxo_financeiro - INFO - [anomalias] VALE3.SA: ['queda_lucro']


2026-11-06 21:39:01,617 - fluxo_financeiro - INFO - [anomalias] detectando sinais para WEGE3.SA


VALE3: 1 sinal(is)
   [queda_lucro] O lucro líquido de VALE3.SA no trimestre encerrado em 2026-09-30 caiu 53% em relação ao mesmo trimestre do ano anterior (R$ 1.25 bi contra R$ 2.69 bi).


2026-11-06 21:39:02,680 - fluxo_financeiro - INFO - [anomalias] WEGE3.SA: nenhum sinal


WEGE3: 0 sinal(is)


No pipeline, quem chama o detector é o nó do YFinance: os sinais vão para o campo `sinais_anomalia` do estado
e também para o fim da mensagem do YFinance, na seção `SINAIS DE ANOMALIA (calculados em código)`.

## 2. Os guards, um por um

Chamamos o `supervisor_node` com estados montados à mão, como se o pipeline estivesse em cada ponto.
Todas as células abaixo param num guard, então **nenhuma chama o LLM**.

In [3]:
from langchain_core.messages import HumanMessage

from fluxo_financeiro.graph.supervisor import supervisor_node

pedido = HumanMessage("Analise a ação VALE3")
dados = HumanMessage("Dados para VALE3.SA: ...", name="YFinance")
empresa = HumanMessage("Notícias da empresa...", name="Analyzer")
setor = HumanMessage("Contexto do setor...", name="Analyzer")
diagnostico = HumanMessage("- Causa identificada: ...", name="Investigator")


def mostrar(titulo, messages, sinais_do_estado=(), loop_count=0):
    estado = {"messages": messages, "loop_count": loop_count, "sinais_anomalia": list(sinais_do_estado)}
    decisao = supervisor_node(estado)
    print(f"{titulo}\n   -> {decisao['next_agent']} (decidido por: {decisao['decidido_por']})")
    for m in decisao.get("messages", []):
        print(f"   tarefa: {m.text[:150]}")
    print()


mostrar("Etapa 2: YFinance coletou, Analyzer não rodou", [pedido, dados])
mostrar("Etapa 3: Analyzer falou uma vez (falta o setor)", [pedido, dados, empresa])
mostrar("Etapa 4: duas passagens do Analyzer + sinais da VALE3", [pedido, dados, empresa, setor], sinais["VALE3"])
mostrar("Investigator já respondeu", [pedido, dados, empresa, setor, diagnostico])
mostrar("Hard stop: loop_count chegou a 6", [pedido], loop_count=6)

2026-11-06 21:39:02,690 - fluxo_financeiro - INFO - [supervisor] inicio — loop_count=0


2026-11-06 21:39:02,691 - fluxo_financeiro - INFO - [supervisor] guard: YFinance coletou → forcando Analyzer (passo 1: noticias da empresa)


2026-11-06 21:39:02,691 - fluxo_financeiro - INFO - [supervisor] inicio — loop_count=0


2026-11-06 21:39:02,692 - fluxo_financeiro - INFO - [supervisor] guard: 1 passagem do Analyzer → forcando Analyzer (etapa 3: setor)


2026-11-06 21:39:02,692 - fluxo_financeiro - INFO - [supervisor] inicio — loop_count=0


2026-11-06 21:39:02,693 - fluxo_financeiro - INFO - [supervisor] guard: sinais de anomalia (queda_lucro) → forcando Investigator


2026-11-06 21:39:02,693 - fluxo_financeiro - INFO - [supervisor] inicio — loop_count=0


2026-11-06 21:39:02,694 - fluxo_financeiro - INFO - [supervisor] guard: Investigator concluiu → FINISH


2026-11-06 21:39:02,694 - fluxo_financeiro - INFO - [supervisor] inicio — loop_count=6


2026-11-06 21:39:02,694 - fluxo_financeiro - WARNING - [supervisor] hard stop — loop_count=6 >= 6


Etapa 2: YFinance coletou, Analyzer não rodou
   -> Analyzer (decidido por: guard)
   tarefa: Busque notícias recentes sobre a empresa no histórico. Use os dados financeiros do YFinance como contexto para identificar o ticker e o nome da empres

Etapa 3: Analyzer falou uma vez (falta o setor)
   -> Analyzer (decidido por: guard)
   tarefa: Busque informações sobre o setor de atuação da empresa (VALE3), usando a indústria informada nos dados do YFinance, e o posicionamento dela nesse seto

Etapa 4: duas passagens do Analyzer + sinais da VALE3
   -> Investigator (decidido por: guard)
   tarefa: Investigue as anomalias detectadas nos números da empresa:
- O lucro líquido de VALE3.SA no trimestre encerrado em 2026-09-30 caiu 53% em relação ao m

Investigator já respondeu
   -> FINISH (decidido por: guard)

Hard stop: loop_count chegou a 6
   -> FINISH (decidido por: hard_stop)



## 3. E quando não há sinal?

Com duas passagens do Analyzer e `sinais_anomalia` vazio (o caso da WEGE3), nenhum guard se aplica e a decisão
vai para o LLM, com a pergunta: *"Um investidor ficaria com dúvida de POR QUE esse número aconteceu?"*.
O notebook 03 mostra esse caso rodando na análise completa.

## O que observar

- Se a VALE3 tiver sinais, a etapa 4 vai para o Investigator **por guard**, e a tarefa dele é montada com as frases dos sinais, com números. Se a lista da VALE3 vier vazia (o yfinance atualizou os trimestres), a etapa 4 cai no LLM.
- Todas as decisões dizem `decidido por: guard` (ou `hard_stop`): é esse campo que o dashboard desenha nas setas.
- A tarefa da etapa 3 cita o ticker (lido do pedido do usuário com `extrair_ticker`) e manda usar a indústria que o YFinance trouxe.
- Os limites (30%, Piotroski 3) foram calibrados com poucos tickers: é uma das limitações conhecidas da v1.